In [10]:
import pandas as pd
import numpy as np

## Essai 1

In [11]:
# A ajouter : load direct depuis http

# Load of movie dataset
movies = pd.read_csv('data_mdl/movies.zip',compression='zip', sep=',', usecols=['movieId', 'title'])

# Split year from title
movies['year'] = movies['title'].str.extract('.*\((.*)\).*',expand = False)
movies['title'] = movies['title'].apply(lambda x: x[:-7])

# Clean of years values
dict_replace = {'1975-1979': '1979',
'2009– ': '2009',
'2007-':'2007', 
'Das Millionenspiel':'1970', 
'Bicicleta, cullera, poma':'2010',
'1983)':'1983'
}

movies['year'] = movies['year'].replace(to_replace=dict_replace) 

# Drop of rows with missing year (17 movies)
movies_without_year = movies[movies['year'].isnull()][['movieId', 'title']]
movies = movies.dropna(axis=0, how='any', subset=['year'])

# Change 'year' type for int
movies['year'] = movies['year'].astype('int')

# Load of links dataset
links = pd.read_csv('data_mdl/links.zip',compression='zip', sep=',')

links = links.drop(['tmdbId'], axis=1)

# Merge : add of imdb Id
df = movies.merge(right=links, left_on='movieId', right_on='movieId', how='left')


print('Nbre de films : ', len(df.movieId.unique()), '\n')
print(df.info())
df.head(5)


Nbre de films :  27261 

<class 'pandas.core.frame.DataFrame'>
Int64Index: 27261 entries, 0 to 27260
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  27261 non-null  int64 
 1   title    27261 non-null  object
 2   year     27261 non-null  int64 
 3   imdbId   27261 non-null  int64 
dtypes: int64(3), object(1)
memory usage: 1.0+ MB
None


,movieId,title,year,imdbId
0,1,Toy Story,1995,114709
1,2,Jumanji,1995,113497
2,3,Grumpier Old Men,1995,113228
3,4,Waiting to Exhale,1995,114885
4,5,Father of the Bride Part II,1995,113041


In [12]:
# Limitation of dataset
df = df[ (df['year'] > 2000) & (df['year'] < 2010) ]

# Save movie list to filter next datasets
list_movieId_2000_2010 = df['movieId'].unique().tolist()
list_imdbId_2000_2010 = df['imdbId'].unique().tolist()


np.savetxt("data_processed/list_movieId_2000_2010.csv" , list_movieId_2000_2010, delimiter =", ", fmt ='% s')
np.savetxt("data_processed/list_imdbId_2000_2010.csv" , list_imdbId_2000_2010 , delimiter =", ",  fmt ='% s')


#print('Nbre de films : ', len(df.movieId.unique()), '\n')
#print(df.info())
#df.head(5)
list_imdbId_2000_2010[:10] 

[218817,
 238948,
 206275,
 237572,
 186589,
 209475,
 178642,
 192111,
 242998,
 120753]

## Essai 2

In [25]:
# Load Imdb dataset
title_basics = pd.read_csv('data_imdb/title.basics.tsv.gz',compression='gzip', sep='\t', usecols=['tconst','primaryTitle','startYear'] , na_values=['\\N', 'nan']) #,  dtype=dict_types

# preprocess of title_crew.tconst to match imdbId from mdl
title_basics['tconst_short'] = [x[-7:] for x in title_basics['tconst']]
title_basics['tconst_short'] = title_basics['tconst_short'].astype('int64')

# Load of links dataset
links = pd.read_csv('data_mdl/links.zip',compression='zip', sep=',')
links = links.drop(['tmdbId'], axis=1)

# Merge
imdb_mdl_mapp = title_basics.merge(right=links, left_on='tconst_short', right_on='imdbId', how='left')
imdb_mdl_mapp = imdb_mdl_mapp.drop(['tconst_short', 'imdbId'], axis=1)


# Drop rows containing NANs
imdb_mdl_mapp = imdb_mdl_mapp.dropna(axis=0)

# Change types
dict_types = {'tconst':object, 'movieId':int, 'primaryTitle':object, 'startYear':int}
imdb_mdl_mapp = imdb_mdl_mapp.astype(dict_types)

# Re-order columns
imdb_mdl_mapp = imdb_mdl_mapp[['tconst', 'movieId', 'primaryTitle', 'startYear' ]]


# Save
imdb_mdl_mapp.to_csv('data_processed/imdb_mdl_mapp.csv')


print('Nbre de films : ', len(imdb_mdl_mapp.movieId.unique()), '\n')
print(imdb_mdl_mapp.info())
imdb_mdl_mapp.head(5)

Nbre de films :  27248 

<class 'pandas.core.frame.DataFrame'>
Int64Index: 41346 entries, 4 to 7237807
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   tconst        41346 non-null  object
 1   movieId       41346 non-null  int64 
 2   primaryTitle  41346 non-null  object
 3   startYear     41346 non-null  int64 
dtypes: int64(2), object(2)
memory usage: 1.6+ MB
None


,tconst,movieId,primaryTitle,startYear
4,tt0000005,95541,Blacksmith Scene,1893
7,tt0000008,88674,Edison Kinetoscopic Record of a Sneeze,1894
9,tt0000010,120869,Leaving the Factory,1895
11,tt0000012,98981,The Arrival of a Train,1896
13,tt0000014,113048,The Waterer Watered,1895


In [24]:
imdb_mdl_mapp[imdb_mdl_mapp['movieId'].isna()==True]

,tconst,movieId,primaryTitle,startYear
